In [ ]:
import pandas as pd

processed_code_pair_df = pd.read_csv('saved_files/mimic4/final_filtered_pair_mimic4_df.csv')
save_prefix='saved_files/mimic4/mimic4'
df_p_code_smoothed = pd.read_csv(f'{save_prefix}_code_marginal_probs_ccs.csv')

In [ ]:
# 4) Optional: simple bar chart (with % on top)
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
bars = plt.bar(cat_counts['pair_category'], cat_counts['count'])

plt.title("Distribution of final evidence-supported code pairs (MIMIC-IV)")
plt.xlabel("Pair category")
plt.ylabel("Number of code pairs")
plt.xticks(rotation=45, ha='right')

# ---- add percentage labels on top of each bar ----
ymax = cat_counts['count'].max()


plt.ylim(0, ymax * 1.12)          # add ~12% headroom (tune 1.10–1.25)
offset = ymax * 0.02              # text gap above bars (tune 0.01–0.03)

for bar, pct in zip(bars, cat_counts['percent']):
    h = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        h + offset,
        f"{pct:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )
# -----------------------------------------------

plt.tight_layout()
plt.savefig('figures/mimic4_bar_plot', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
CANONICAL_LABELS_WITH_DESC = {
    frozenset({"dx","dx"}): {
        "causes": "codeA is an etiologic cause of codeB (direct causal/pathophysiologic link).",
        "risk_factor_for": "codeA increases the likelihood of codeB (epidemiologic association; not necessarily causal).",
        "leads_to": "codeA typically precedes codeB as a downstream condition or stage (temporal progression without requiring strict causality).",
        "complicates": "codeA occurs as a complication during the course of codeB (arises secondary to codeB or its treatment).",
        "co_occurs_with": "codeA and codeB are frequently observed together without implied direction or clear causality (contextual association).",
        "no_significant_relation": "Clinical prior and sufficient data indicate no clinically meaningful association between codeA and codeB",
        "cannot_decide": "Insufficient/conflicting evidence; abstain from assigning a relation."
    },
    frozenset({"rx","dx"}): {
        "treats": "codeA treats or manages codeB (therapeutic use).",
        "prevents": "codeA reduces risk or recurrence of codeB (prophylaxis).",
        "causes_adverse_event": "codeA may induce codeB as an adverse reaction.",
        "contraindicated_for": "codeA should be avoided when codeB is present.",
        "co_occurs_with": "codeA and codeB often co-occur without clear causal link.",
        "no_significant_relation": "Clinical prior and sufficient data indicate no clinically meaningful association between codeA and codeB.",
        "cannot_decide": "Insufficient/conflicting evidence; abstain from assigning a relation."
    },
    frozenset({"px","dx"}): {
        "diagnostic_of": "codeA is performed to diagnose, confirm, or rule in/out codeB (diagnostic evaluation).",
        "treats": "codeA is a procedure used to treat, correct, or palliate codeB (therapeutic intervention).",
        "monitors": "codeA is performed to monitor, follow, or assess disease activity or treatment response for codeB.",
        "contraindicated_for": "codeA should be avoided when codeB is present due to safety or unfavorable risk–benefit.",
        "co_occurs_with": "codeA and codeB frequently appear together in data without a clear diagnostic, therapeutic, or causal link (contextual association).",
        "no_significant_relation": "Clinical prior and sufficient data indicate no clinically meaningful association between codeA and codeB.",
        "cannot_decide": "Insufficient/conflicting evidence; abstain from assigning a relation."
    },
    frozenset({"rx","rx"}):{
        "co_prescribed_with": "codeA and codeB are intentionally prescribed together in practice.",
        "contraindicated_with": "Concomitant use of codeA and codeB is contraindicated due to serious safety risk.",
        "interacts_with": "codeA and codeB have a clinically meaningful drug–drug interaction (PK and/or PD) that may impact safety or efficacy.",
        "substitute_for": "codeA is commonly used as a therapeutic alternative to codeB for similar indications (typically not co-prescribed).",
        "combination_therapy_with": "codeA and codeB are used together as established combination therapy.",
        "co_occurs_with": "codeA and codeB frequently appear together in data without a known therapeutic relationship or interaction.",
        "no_significant_relation": "Clinical prior and sufficient data indicate no clinically meaningful association between codeA and codeB.",
        "cannot_decide": "Insufficient/conflicting evidence; abstain from assigning a relation."
    },
    frozenset({"px","px"}): {
        "sequential_care": "codeA is typically followed by codeB as the next procedural step (ordered workflow; not necessarily causal).",
        "prerequisite_for": "codeA is commonly required/preparatory for performing codeB (e.g., access, imaging guidance, setup).",
        "alternative_to": "codeA and codeB are procedural alternatives for a similar clinical purpose (mutually substitutable options).",
        "performed_same_session": "codeA and codeB are commonly performed during the same procedural session/time block within an episode (intentional bundling).",
        "co_occurs_with": "codeA and codeB occur within the same episode without implied order or intentional bundling (contextual association).",
        "no_significant_relation": "Clinical prior and sufficient data indicate no clinically meaningful association between codeA and codeB.",
        "cannot_decide": "Insufficient evidence: no strong clinical prior and statistical signals are low-support, unstable, or conflicting; abstain from assigning a relation."
    },
   frozenset({"px","rx"}):{
        "requires_medication": "codeA routinely requires administration of codeB as part of the procedure (e.g., sedation, anesthesia, anticoagulation).",
       "premedication_for": "codeB is typically given before codeA to enable or optimize the procedure (e.g., anxiolysis, antibiotic prophylaxis).",
       "post_procedure_medication_for": "codeB is commonly given after codeA for recovery, prophylaxis, or symptom control (e.g., analgesia, anticoagulation).",
       "prophylaxis_with": "codeB is used to prevent complications associated with codeA (e.g., peri-procedural antibiotics, DVT prophylaxis).",
       "adjunct_medication_for": "codeB is an adjunct given with codeA to improve efficacy, safety, or tolerability (non-essential but common).",
        "contraindicates_medication": "When codeA is planned or present, codeB should be avoided due to safety or risk–benefit concerns.",
        "co_occurs_with": "codeA and codeB are observed together in data without a clear procedural rationale or causal link (contextual association).",
        "no_significant_relation": "Clinical prior and sufficient data indicate no clinically meaningful association between codeA and codeB.",
        "cannot_decide": "Insufficient/conflicting evidence; abstain from assigning a relation."
        }
}


# === Which labels are symmetric (order-invariant) for each pair type ===
SYMMETRIC_LABELS = {
    frozenset({"dx","dx"}): ["co_occurs_with"],
    frozenset({"dx","rx"}): ["co_occurs_with"],
    frozenset({"dx","px"}): ["co_occurs_with"],

    # Rx–Rx: all symmetric; added the missing comma and included contraindicated_with
    frozenset({"rx","rx"}): [
        "interacts_with",
        "co_prescribed_with",
        "combination_therapy_with",
        "substitute_for",
        "co_occurs_with",
        "contraindicated_with"
    ],

    # Px–Px: standardized label name to performed_same_session
    frozenset({"px","px"}): ["alternative_to", "performed_same_session", "co_occurs_with"],

    # Px–Rx: only co_occurs_with is symmetric (adjunct_* is directional in your taxonomy)
    frozenset({"px","rx"}): ["co_occurs_with"]
}

INVERSE_RELATION_CANONICAL = {
    # --- dx–dx (directional) ---
    (frozenset({"dx","dx"}), "causes"):            (frozenset({"dx","dx"}), "caused_by"),
    (frozenset({"dx","dx"}), "caused_by"):         (frozenset({"dx","dx"}), "causes"),

    (frozenset({"dx","dx"}), "risk_factor_for"):   (frozenset({"dx","dx"}), "has_risk_factor"),
    (frozenset({"dx","dx"}), "has_risk_factor"):   (frozenset({"dx","dx"}), "risk_factor_for"),

    (frozenset({"dx","dx"}), "leads_to"):          (frozenset({"dx","dx"}), "results_from"),
    (frozenset({"dx","dx"}), "results_from"):      (frozenset({"dx","dx"}), "leads_to"),

    (frozenset({"dx","dx"}), "complicates"):       (frozenset({"dx","dx"}), "complication_of"),
    (frozenset({"dx","dx"}), "complication_of"):   (frozenset({"dx","dx"}), "complicates"),

    # (Symmetric dx–dx like co_occurs_with → self-inverse; omit)

    # --- rx–dx (drug ↔ condition, directional) ---
    (frozenset({"rx","dx"}), "treats"):                (frozenset({"rx","dx"}), "treated_by"),
    (frozenset({"rx","dx"}), "treated_by"):            (frozenset({"rx","dx"}), "treats"),

    (frozenset({"rx","dx"}), "prevents"):              (frozenset({"rx","dx"}), "prevented_by"),
    (frozenset({"rx","dx"}), "prevented_by"):          (frozenset({"rx","dx"}), "prevents"),

    (frozenset({"rx","dx"}), "causes_adverse_event"):  (frozenset({"rx","dx"}), "adverse_event_of"),
    (frozenset({"rx","dx"}), "adverse_event_of"):      (frozenset({"rx","dx"}), "causes_adverse_event"),

    (frozenset({"rx","dx"}), "contraindicated_for"):   (frozenset({"rx","dx"}), "has_contraindication"),
    (frozenset({"rx","dx"}), "has_contraindication"):  (frozenset({"rx","dx"}), "contraindicated_for"),

    # (Symmetric rx–dx like co_occurs_with → self-inverse; omit)

    # --- dx–px (condition ↔ procedure, directional) ---
    (frozenset({"dx","px"}), "indicates_procedure"):   (frozenset({"dx","px"}), "diagnostic_of"),
    (frozenset({"dx","px"}), "diagnostic_of"):         (frozenset({"dx","px"}), "indicates_procedure"),

    (frozenset({"dx","px"}), "treated_by_procedure"):  (frozenset({"dx","px"}), "treats"),
    (frozenset({"dx","px"}), "treats"):                (frozenset({"dx","px"}), "treated_by_procedure"),

    (frozenset({"dx","px"}), "monitored_by"):          (frozenset({"dx","px"}), "monitors"),
    (frozenset({"dx","px"}), "monitors"):              (frozenset({"dx","px"}), "monitored_by"),

    (frozenset({"dx","px"}), "contraindicates_procedure"): (frozenset({"dx","px"}), "contraindicated_for"),
    (frozenset({"dx","px"}), "contraindicated_for"):       (frozenset({"dx","px"}), "contraindicates_procedure"),

    # (Symmetric dx–px like co_occurs_with → self-inverse; omit)

    # --- px–px (procedure ↔ procedure, directional) ---
    (frozenset({"px","px"}), "sequential_care"):      (frozenset({"px","px"}), "preceded_by_step"),
    (frozenset({"px","px"}), "preceded_by_step"):     (frozenset({"px","px"}), "sequential_care"),

    (frozenset({"px","px"}), "prerequisite_for"):     (frozenset({"px","px"}), "has_prerequisite"),
    (frozenset({"px","px"}), "has_prerequisite"):     (frozenset({"px","px"}), "prerequisite_for"),

    # (Symmetric px–px like alternative_to, performed_same_session, co_occurs_with → self-inverse; omit)

    # --- px–rx (procedure ↔ drug, directional) ---
    (frozenset({"px","rx"}), "requires_medication"):          (frozenset({"px","rx"}), "required_for"),
    (frozenset({"px","rx"}), "required_for"):                 (frozenset({"px","rx"}), "requires_medication"),

    (frozenset({"px","rx"}), "premedication_for"):            (frozenset({"px","rx"}), "is_premedication_for"),
    (frozenset({"px","rx"}), "is_premedication_for"):         (frozenset({"px","rx"}), "premedication_for"),

    (frozenset({"px","rx"}), "post_procedure_medication_for"): (frozenset({"px","rx"}), "is_post_procedure_medication_for"),
    (frozenset({"px","rx"}), "is_post_procedure_medication_for"): (frozenset({"px","rx"}), "post_procedure_medication_for"),

    (frozenset({"px","rx"}), "prophylaxis_with"):             (frozenset({"px","rx"}), "is_prophylaxis_for"),
    (frozenset({"px","rx"}), "is_prophylaxis_for"):           (frozenset({"px","rx"}), "prophylaxis_with"),

    (frozenset({"px","rx"}), "adjunct_medication_for"):       (frozenset({"px","rx"}), "is_adjunct_medication_for"),
    (frozenset({"px","rx"}), "is_adjunct_medication_for"):    (frozenset({"px","rx"}), "adjunct_medication_for"),

    (frozenset({"px","rx"}), "contraindicates_medication"):   (frozenset({"px","rx"}), "contraindicated_for"),
    (frozenset({"px","rx"}), "contraindicated_for"):          (frozenset({"px","rx"}), "contraindicates_medication"),

    # (Symmetric px–rx like co_occurs_with → self-inverse; omit)
}


In [ ]:
unique_key_set=[]
for key in CANONICAL_LABELS_WITH_DESC.keys():
    unique_key_set.extend(list(CANONICAL_LABELS_WITH_DESC[key].keys()))

unique_key_set= set(unique_key_set)

len(unique_key_set)

In [ ]:
#util
# robust formatter: None/NaN/Inf -> "NA"
import math
def fmt(x, nd=4):
    "robust formatter: None/NaN/Inf -> NA"
    if x is None:
        return "NA"
    try:
        xf = float(x)
        if math.isnan(xf) or math.isinf(xf):
            return "NA"
        return f"{xf:.{nd}f}"
    except Exception:
        return "NA"

# =====================================
# 2) Prompt builder (unordered pair, 12/21)
# =====================================

def build_prompt(pair_info, dataset_info, stats, CANONICAL_LABELS_WITH_DESC,SYMMETRIC_LABELS):
    """
    Build a single prompt for an unordered pair (canonical code1/code2) that includes both
    12 and 21 evidence (shown as NA if missing). The model must output exactly ONE decision:
    a canonical label (from the unordered label set) and an orientation in {12, 21, symmetric}.

    pair_info: dict with keys [code1_id, code1_name, code1_parent_id, code1_parent_name, type1,
                               code2_id, code2_name, code2_parent_id, code2_parent_name, type2,
                               type1_dis, type2_dis, p_code1, p_code2]
    dataset_info: total_codes, dx_count, rx_count, px_count, avg_visits (optional), domain_desc
    stats keys (any may be missing/None):
      - Co (12):  P_c2_given_c1_co_12, RR_12, PMI_12, p_support_co_12, pmi_support_co_12
      - Co (21):  P_c1_given_c2_co_21, RR_21, PMI_21, p_support_co_21, pmi_support_co_21
      - Temp(12): P_c2_given_c1_temp_12, TemporalPMI_12, p_support_temp_12, pmi_support_temp_12
      - Temp(21): P_c1_given_c2_temp_21, TemporalPMI_21, p_support_temp_21, pmi_support_temp_21
    """
    stats = stats or {}
    # ---- Canonical (unordered) label set retrieval ----
    tset = frozenset({pair_info['type1'], pair_info['type2']})
    try:
        candidates = CANONICAL_LABELS_WITH_DESC[tset]
    except KeyError:
        raise KeyError(
            f"Unknown unordered type-set {set(tset)}. "
            f"Valid keys: { [set(k) for k in CANONICAL_LABELS_WITH_DESC.keys()] }"
        )
    #allowed_labels = list(candidates.keys())
    symmetric_labels = SYMMETRIC_LABELS.get(tset, ["co_occurs_with"])


    co12 = {
        "P":   fmt(stats.get("P_c2_given_c1_co_12")),
        "PMI": fmt(stats.get("PMI_12")),
        "sp_P": fmt(stats.get("p_support_co_12")),
        "sp_PMI": fmt(stats.get("pmi_support_co_12")),
    }
    co21 = {
        "P":   fmt(stats.get("P_c1_given_c2_co_21")),
        "PMI": fmt(stats.get("PMI_21")),
        "sp_P": fmt(stats.get("p_support_co_21")),
        "sp_PMI": fmt(stats.get("pmi_support_co_21")),
    }
    tp12 = {
        "tP":    fmt(stats.get("P_c2_given_c1_temp_12")),
        "tPMI": fmt(stats.get("TemporalPMI_12")),
        "sp_tP":  fmt(stats.get("p_support_temp_12")),
        "sp_tPMI":  fmt(stats.get("pmi_support_temp_12")),
    }
    tp21 = {
        "tP":    fmt(stats.get("P_c1_given_c2_temp_21")),
        "tPMI": fmt(stats.get("TemporalPMI_21")),
        "sp_tP":  fmt(stats.get("p_support_temp_21")),
        "sp_tPMI":  fmt(stats.get("pmi_support_temp_21")),
    }

    # Build candidate relationship list with descriptions
    rel_block = "\n".join(
        [f"- {rel}: {desc}" for rel, desc in candidates.items()]
    )


    metrics_glossary = (
        "### Metrics Glossary\n\n"
        "#### Pair & Direction\n"
        "- Canonical: code1 ≤ code2. **12 = code1→code2**, **21 = code2→code1**.\n"
        "- Co-visit stores both (a,b) and (b,a).\n\n"
        "#### Co-visit (same visit; duplicates removed)\n"
        "- Counts: count_visit(x), count_visit(x,y), N_visit.\n"
        "- **P(y|x)** = (count_visit(x,y)+α)/(count_visit(x)+α·V), α=0.01, V=#codes. **sp_P**=count_visit(x).\n"
        "- Marginal: P(y)=count_visit(y)/N_visit.\n"
        "- **PMI(x,y)** = log2([count_visit(x,y)·N_visit]/[count_visit(x)·count_visit(y)]). **sp_PMI**=count_visit(x,y).\n\n"
        "#### Temporal (next visit)\n"
        "- Transitions: t→t+1.\n"
        "- Counts: count_trans(x→y), source(x), target(y); totals T_trans, T_source.\n"
        "- **P_next(y|x)** = (count_trans(x→y)+α)/(source(x)+α·V), α=0.01, V=|sources∪targets|. **sp_P**=source(x).\n"
        "- Marginals: p(x→y)=count_trans(x→y)/T_trans; p(x→*)=source(x)/T_trans; p(*→y)=target(y)/T_trans.\n"
        "- **tPMI(x→y)** = log2[p(x→y)/(p(x→*)·p(*→y))]. **sp_PMI**=count_trans(x→y).\n"
    )


    decision_rules = f"""
### Decision Rules
- Use your clinical knowledge first; then the statistics provided.
- If evidence is weak or conflicting, choose a conservative label.
- If edge_orientation = `12` or `symmetric` or 'none': `KG_triple` must be ["{pair_info['code1_id']}", "<predicted_relationship>", "{pair_info['code2_id']}"]. If edge_orientation = `21`: `KG_triple` must be ["{pair_info['code2_id']}", "<predicted_relationship>", "{pair_info['code1_id']}"].
- Base-rate awareness: If P(code1) or P(code2) ≥ 0.60 (ubiquitous) or ≤ 0.01 (rare), interpret P/PMI/tPMI conservatively
- Return **exactly one** relationship decision per pair.
    """


    confidence_rubric = (
        "### Confidence Calibration\n"
        "- 90–100: strong clinical prior AND high-support stats; co & temporal align.\n"
        "- 70–89: clear prior AND at least one strong statistical signal.\n"
        "- 50–69: plausible but limited/conflicting evidence.\n"
        "- <50: weak/contradictory evidence.\n"
    )

    p_code1 = fmt(pair_info.get("p_code1"))
    p_code2 = fmt(pair_info.get("p_code2"))
    unordered_type = f"{pair_info['type1']}↔{pair_info['type2']}"

    # Final prompt
    prompt = f"""
### Task
You are a medical reasoning expert. Infer the **most plausible semantic relationship** between two clinical concepts using medical knowledge first, then statistical signals (co-occurrence and temporal evidence).

### Dataset
- Unique code counts: Total: {dataset_info['total_codes']}; dx (CCS)={dataset_info['dx_count']}, rx (ATC)={dataset_info['rx_count']}, px (CCS)={dataset_info['px_count']}
- Domain: {dataset_info['domain_desc']}

### Code Pair (canonical order; your output will reference 'code1' and 'code2' below)
- code1: {pair_info['code1_id']} — {pair_info['code1_name']} — type={pair_info['type1_dis']} | P(code1)={p_code1}
- code2: {pair_info['code2_id']} — {pair_info['code2_name']} — type={pair_info['type2_dis']} | P(code2)={p_code2}
> `P(code)` = visit-level marginal frequency in this dataset.

{metrics_glossary}
### Statistical Signals (supporting evidence)
## Co-occurrence (same visit)
- Direction 12 (code1→code2): P={co12['P']}, PMI={co12['PMI']}, supports: P={co12['sp_P']}, PMI={co12['sp_PMI']}
- Direction 21 (code2→code1): P={co21['P']}, PMI={co21['PMI']}, supports: P={co21['sp_P']}, PMI={co21['sp_PMI']}

## Temporal (next visit; directional)
- Direction 12 (code1→code2): P_next={tp12['tP']}, tPMI={tp12['tPMI']}, supports: P={tp12['sp_tP']}, tPMI={tp12['sp_tPMI']}
- Direction 21 (code2→code1): P_next={tp21['tP']}, tPMI={tp21['tPMI']}, supports: P={tp21['sp_tP']}, tPMI={tp21['sp_tPMI']}

### Candidate Labels (for {unordered_type})
{rel_block}

{confidence_rubric}
{decision_rules}

### Output (JSON only; no prose)
{{
  "predicted_relationship": "<exactly ONE of AllowedLabels>",
  "KG_triple": ["{{head_id}}", "<predicted_relationship>", "{{tail_id}}"],
  "confidence": <integer 0-100>,
}}

- Strict output rule: Return ONLY ONE **valid JSON only**; no extra text.
"""
    return prompt


In [ ]:
import pandas as pd
from typing import Dict, Optional, List, Tuple
from pyhealth.medcode import InnerMap
import os, json, time, hashlib, re
from openai import OpenAI, APIStatusError
from tqdm import tqdm
# -----------------------
# Config / Model client
# -----------------------


# -----------------------
# Utilities
# -----------------------
def fallback_name(val: Optional[str], code: str) -> str:
    try:
        if val is None: return code
        if isinstance(val, float) and pd.isna(val): return code
        s = str(val).strip()
        return s if s else code
    except Exception:
        return code

atc = InnerMap.load("ATC")
def atc_name(code_unprefixed: str) -> str:
    try:
        res = atc.lookup(code_unprefixed)
        if res is None: return code_unprefixed
        if isinstance(res, (list, tuple)): return str(res[0]) if res else code_unprefixed
        return str(res)
    except Exception:
        return code_unprefixed





atc = InnerMap.load("ATC")
if "code_modified" not in df_p_code_smoothed.columns:
    df_p_code_smoothed['code_modified'] = df_p_code_smoothed['code'].apply(lambda  x: x[3:])

df_ccs_px_labels = pd.read_csv("datasets/CCS_maping/ccs_proc_category_labels_clean.csv",
                               dtype=str, keep_default_na=False, na_filter=False, engine="python")
df_ccs_dx_labels = pd.read_csv("datasets/CCS_maping/ccs_category_labels_clean.csv",
                               dtype=str, keep_default_na=False, na_filter=False, engine="python")
dx_name_map = df_ccs_dx_labels.set_index('level_code')['level_label']
px_name_map = df_ccs_px_labels.set_index('level_code')['level_label']
P_code_map = df_p_code_smoothed.set_index('code_modified')['P(code)']

dataset_info = {
    "total_codes": 1394, "dx_count": 562, "rx_count": 510, "px_count": 322,"domain_desc": "ICU+inpatient EHR; visit= one hospitalization (with an admit and discharge time)"
}


def row_to_prompt(row: pd.Series) -> str:
    # pair info
    code1 = row['code1'][3:]
    type1 = row['type1']
    code2 = row['code2'][3:]
    type2 = row['type2']

    if type1 == 'dx':
        type1_dis = 'CCS diagnosis code'
        code1_name = fallback_name(dx_name_map.get(code1), code1)
    elif type1 == 'px':
        type1_dis = 'CCS procedure code'
        code1_name = fallback_name(px_name_map.get(code1), code1)
    else:  # rx
        type1_dis = 'ATC drug code'
        code1_name = atc_name(code1)

    if type2 == 'dx':
        type2_dis = 'CCS diagnosis code'
        code2_name = fallback_name(dx_name_map.get(code2), code2)
    elif type2 == 'px':
        type2_dis = 'CCS procedure code'
        code2_name = fallback_name(px_name_map.get(code2), code2)
    else:  # rx
        type2_dis = 'ATC drug code'
        code2_name = atc_name(code2)


    pair_info = {
    "code1_id": code1, "code1_name": code1_name, "type1": type1,"type1_dis": type1_dis, 'p_code1' : P_code_map.get(code1),
    "code2_id": code2, "code2_name": code2_name ,"type2": type2, "type2_dis": type2_dis, 'p_code2' : P_code_map.get(code2)
    }

    # stats (use .get to be robust)
    stats: Dict[str, Optional[float]] = {
        # co 12
        "P_c2_given_c1_co_12":  row.get("P_c2_given_c1_co_12"),
        "p_support_co_12":      row.get("p_support_co_12"),
        "PMI_12":               row.get("PMI_12"),
        "pmi_support_co_12":    row.get("pmi_support_co_12"),
        #"RR_12":                row.get("RR_12"),
        # co 21
        "P_c1_given_c2_co_21":  row.get("P_c1_given_c2_co_21"),
        "p_support_co_21":      row.get("p_support_co_21"),
        "PMI_21":               row.get("PMI_21"),
        "pmi_support_co_21":    row.get("pmi_support_co_21"),
        #"RR_21":                row.get("RR_21"),
        # temp 12
        "P_c2_given_c1_temp_12":row.get("P_c2_given_c1_temp_12"),
        "p_support_temp_12":    row.get("p_support_temp_12"),
        "TemporalPMI_12":       row.get("TemporalPMI_12"),
        "pmi_support_temp_12":  row.get("pmi_support_temp_12"),
        # temp 21
        "P_c1_given_c2_temp_21":row.get("P_c1_given_c2_temp_21"),
        "p_support_temp_21":    row.get("p_support_temp_21"),
        "TemporalPMI_21":       row.get("TemporalPMI_21"),
        "pmi_support_temp_21":  row.get("pmi_support_temp_21"),
    }

    return pair_info, stats, build_prompt(pair_info=pair_info, dataset_info=dataset_info, stats=stats, CANONICAL_LABELS_WITH_DESC=CANONICAL_LABELS_WITH_DESC,SYMMETRIC_LABELS=SYMMETRIC_LABELS)




In [ ]:
def prompt_hash(prompt: str) -> str:
    return hashlib.sha1(prompt.encode("utf-8")).hexdigest()


def config_signature(model: str, metrics_cfg: dict) -> str:
    blob = json.dumps({"model": model, "metrics": metrics_cfg}, sort_keys=True)
    return hashlib.sha1(blob.encode("utf-8")).hexdigest()

In [ ]:
import json, math, os
from pathlib import Path
import pandas as pd

MODEL_PRIMARY = "gpt-5"
OUT_DIR = Path("saved_files/mimic4/KG_openai/batch/gpt5_mini")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_SIG = config_signature(
    model=MODEL_PRIMARY,
    metrics_cfg={"alpha": 0.01, "min_p_support": 5, "min_pmi_support": 5, "temporal_mode": "next"},
)

# --- Your schema for strict JSON (Responses API) ---
STRICT_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "clinical_relation_decision",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "predicted_relationship": {"type": "string"},
                "KG_triple": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 3,
                    "maxItems": 3
                },
                "confidence": {"type": "integer", "minimum": 0, "maximum": 100}
            },
            # required keys reflect the new compact schema (edge_orientation removed)
            "required": ["predicted_relationship", "KG_triple", "confidence"],
            "additionalProperties": False
        }
    }
}



FORMAT_OBJ = {"type": "json_schema", **STRICT_SCHEMA["json_schema"]}

def estimate_bytes(s: str) -> int:
    return len(s.encode("utf-8"))

def make_custom_id(pair_info: dict, prompt: str) -> str:
    return f"{pair_info['code1_id']}__{pair_info['code2_id']}__{prompt_hash(prompt)}"

def write_batch_inputs(processed_df: pd.DataFrame, max_reqs=18000, max_bytes=150*1024*1024):
    """
    Splits the dataset into multiple JSONL files under OUT_DIR/input_part_*.jsonl
    and writes a sidecar CSV for mapping custom_id back to metadata.
    """
    part, count, bytes_used = 0, 0, 0
    fout, map_rows = None, []

    def new_part():
        nonlocal part, count, bytes_used, fout, map_rows
        if fout: fout.close()
        part += 1
        count, bytes_used = 0, 0
        map_rows = []
        return open(OUT_DIR / f"input_part_{part:03d}.jsonl", "w", encoding="utf-8")

    fout = new_part()

    for row in processed_df.itertuples(index=False):
        pair_info, stats, prompt = row_to_prompt(pd.Series(row._asdict()))
        req_body = {
            "model": MODEL_PRIMARY,
            "input": [{"role": "user", "content": prompt}],
            "text": {"format": FORMAT_OBJ}
        }
        custom_id = make_custom_id(pair_info, prompt)
        line_obj = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/responses",
            "body": req_body
        }
        line = json.dumps(line_obj, ensure_ascii=False) + "\n"
        line_bytes = estimate_bytes(line)

        # rotate if limits would be exceeded
        if count >= max_reqs or (bytes_used + line_bytes) > max_bytes:
            # flush sidecar
            pd.DataFrame(map_rows).to_csv(OUT_DIR / f"input_part_{part:03d}_map.csv", index=False)
            fout.close()
            fout = new_part()

        fout.write(line)
        count += 1
        bytes_used += line_bytes

        map_rows.append({
            "custom_id": custom_id,
            "code1_id": pair_info["code1_id"],
            "code2_id": pair_info["code2_id"],
            "type1": pair_info["type1"], "type2": pair_info["type2"],
            "prompt_id": prompt_hash(prompt),
            "stats_json": json.dumps(stats),
            "model_id": MODEL_PRIMARY,
            "config_sig": CONFIG_SIG
        })

    # finalize last part
    if map_rows:
        pd.DataFrame(map_rows).to_csv(OUT_DIR / f"input_part_{part:03d}_map.csv", index=False)
        fout.close()

# run once to produce JSONLs
write_batch_inputs(processed_code_pair_df)


In [ ]:
import json

def check_jsonl_file(path):
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
            except Exception as e:
                print(f"Invalid JSON at line {i}: {e}")
                return False
    print("JSONL file is structurally valid.")
    return True

# Example:
check_jsonl_file("saved_files/mimic4/KG_openai/batch/gpt5_mini/input_part_001.jsonl")


def check_required_fields(path):
    required_keys = {"custom_id", "method", "url", "body"}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            obj = json.loads(line)
            missing = required_keys - obj.keys()
            if missing:
                print(f"Line {i} missing keys: {missing}")
                return False
    print("All records have required keys.")
    return True
check_required_fields("saved_files/mimic4/KG_openai/batch/gpt5_mini/input_part_001.jsonl")


from pathlib import Path

# Set your batch directory path
DIR = Path("saved_files/mimic4/KG_openai/batch/gpt5_mini")
total_lines = 0
for fname in DIR.glob("input_part_*.jsonl"):
    with open(fname, "r", encoding="utf-8") as f:
        n_lines = sum(1 for _ in f)
        total_lines+=n_lines
    print(f"{fname.name}: {n_lines} queries")
print('total number of queries:',total_lines)

In [ ]:
import os
import time
import json
import math
import glob
from pathlib import Path
from typing import List
import pandas as pd
from openai import OpenAI


OUT_DIR = Path("saved_files/mimic4/KG_openai/batch/gpt5_mini")


api_key = "s----"
client = OpenAI(api_key=api_key)


def submit_one_batch(client, path_jsonl: Path, completion_window="24h"):
    # Upload file safely with context manager
    with open(path_jsonl, "rb") as f:
        up = client.files.create(file=f, purpose="batch")
    # Create batch job
    job = client.batches.create(
        input_file_id=up.id,
        endpoint="/v1/responses",
        completion_window=completion_window
    )
    return job
from typing import List, Dict, Any, Optional

# Helper to get job id robustly (SDK may return object or dict)
def job_id_of(job) -> str:
    if job is None:
        return ""
    if isinstance(job, dict):
        return job.get("id") or job.get("job_id") or str(job)
    # try attribute
    return getattr(job, "id", getattr(job, "jobId", None) or str(job))

def list_and_submit_all(OUT_DIR,client):
    jobs = []
    map_batch_id_jsan={}
    for p in sorted(OUT_DIR.glob("input_part_*.jsonl")):
        print(f"submitting file {p} ...")
        job = submit_one_batch(client=client, path_jsonl=p)
        map_batch_id_jsan[job_id_of(job)] = p
        jobs.append(job)
        print(f"job for {p} submitted!")
    return jobs, map_batch_id_jsan




In [ ]:
from typing import List, Dict, Any, Optional

# Helper to get job id robustly (SDK may return object or dict)
def job_id_of(job) -> str:
    if job is None:
        return ""
    if isinstance(job, dict):
        return job.get("id") or job.get("job_id") or str(job)
    # try attribute
    return getattr(job, "id", getattr(job, "jobId", None) or str(job))

# Map of statuses considered terminal
TERMINAL_STATUSES = {"completed", "failed", "canceled", "expired"}

def get_job_status(job_id: str) -> Dict[str, Any]:
    """
    Retrieve job metadata from the Batch API.
    Returns dictionary with at least: {"id": job_id, "status": <str>, ...}
    """
    try:
        j = client.batches.retrieve(job_id)
        # Convert to dict-like for easier handling; SDK objects may be like SimpleNamespace
        jd = {}
        # try common attributes
        jd["id"] = getattr(j, "id", None) or j.get("id") if isinstance(j, dict) else job_id
        jd["status"] = getattr(j, "status", None) or (j.get("status") if isinstance(j, dict) else None)
        jd["input_file_id"] = getattr(j, "input_file_id", None) or (j.get("input_file_id") if isinstance(j, dict) else None)
        jd["output_file_id"] = getattr(j, "output_file_id", None) or (j.get("output_file_id") if isinstance(j, dict) else None)
        jd["created_at"] = getattr(j, "created_at", None) or (j.get("created_at") if isinstance(j, dict) else None)
        jd["updated_at"] = getattr(j, "updated_at", None) or (j.get("updated_at") if isinstance(j, dict) else None)
        # include full raw object as fallback if you want to inspect
        jd["_raw"] = j
        jd['file'] = map_batch_id_jsan.get(jd["id"], None)
        return jd
    except Exception as e:
        return {"id": job_id, "status": "error", "error": str(e), "file": map_batch_id_jsan.get(job_id, None)}

def summarize_jobs(jobs: List[Any]) -> List[Dict[str, Any]]:
    """
    Quick summary for a list of jobs (the objects returned by submit).
    Returns list of dicts containing id/status/output_file_id.
    """
    out = []
    for job in jobs:
        jid = job_id_of(job)
        if not jid:
            out.append({"id": None, "status": "invalid", "error": "no id"})
            continue
        jd = get_job_status(jid)
        out.append(jd)
    return out

def print_job_table(job_summaries: List[Dict[str, Any]]):
    print(f"{'job_id':36}  {'status':10}  {'output_file_id':20}  {'error(if any)':36} {'file':20}")
    print("-" * 100)
    for s in job_summaries:
        print(f"{str(s.get('id')):36}  {str(s.get('status')):10}  {str(s.get('output_file_id') or ''):20}  {s.get('error','')}  {s.get('file','')}")


In [ ]:
import os
from pathlib import Path
from typing import Optional, Tuple

def _write_file_from_content(out_path: Path, content):
    """
    Write content to out_path. content might be:
      - a file-like object with .read()
      - bytes
      - str
    """
    if hasattr(content, "read"):
        data = content.read()
        mode = "wb" if isinstance(data, (bytes, bytearray)) else "w"
        with open(out_path, mode) as f:
            f.write(data)
    elif isinstance(content, (bytes, bytearray)):
        with open(out_path, "wb") as f:
            f.write(content)
    else:
        # fallback: write str
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(str(content))

def _download_file_by_id(file_id: str, out_path: Path):
    """
    Try multiple SDK methods to fetch a file resource reliably.
    """
    # 1) retrieve metadata (may raise if file missing)
    meta = client.files.retrieve(file_id)

    # 2) try .content(...)
    try:
        content = client.files.content(meta.id)
        _write_file_from_content(out_path, content)
        return out_path
    except Exception:
        # 3) fallback to client.files.download(...) if available
        try:
            content = client.files.download(meta.id)
            _write_file_from_content(out_path, content)
            return out_path
        except Exception as e:
            # 4) last resort: maybe meta has a url field to GET
            url = getattr(meta, "url", None) or (meta.get("url") if isinstance(meta, dict) else None)
            if url:
                import requests
                r = requests.get(url, headers={"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY','')}"})
                r.raise_for_status()
                out_path.write_bytes(r.content)
                return out_path
            raise RuntimeError(f"Unable to download file {file_id}: {e}")

def download_job_output(out_dir, batch_id):
    """
    Download job output and error files if present.
    Returns (output_path_or_None, error_path_or_None).
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    job = client.batches.retrieve(batch_id)
    status = getattr(job, "status", job.get("status") if isinstance(job, dict) else None)
    print(f"Job {batch_id} status: {status}")

    output_path = None
    error_path = None

    output_file_id = getattr(job, "output_file_id", None) or (job.get("output_file_id") if isinstance(job, dict) else None)
    error_file_id  = getattr(job, "error_file_id", None)  or (job.get("error_file_id") if isinstance(job, dict) else None)

    if output_file_id:
        try:
            output_path = out_dir / f"output_{batch_id}.jsonl"
            print(f"Downloading output_file_id={output_file_id} -> {output_path}")
            _download_file_by_id(output_file_id, output_path)
        except Exception as e:
            print(f"Failed to download output file {output_file_id}: {e}")
            output_path = None
    else:
        print("No output_file_id present on job.")

    if error_file_id:
        try:
            error_path = out_dir / f"error_{batch_id}.jsonl"
            print(f"Downloading error_file_id={error_file_id} -> {error_path}")
            _download_file_by_id(error_file_id, error_path)
        except Exception as e:
            print(f"Failed to download error file {error_file_id}: {e}")
            error_path = None
    else:
        print("No error_file_id present on job.")

    return output_path, error_path



In [ ]:
from typing import Dict, List, Tuple, Optional, FrozenSet

def validate_llm_output(
    obj: dict,
    *,
    tset: FrozenSet[str],           # e.g., frozenset({"dx","px"})
    code1_id: str,
    code2_id: str,
    allowed_labels: List[str],
) -> Tuple[bool, dict, List[str]]:
    """
    Validate compact LLM JSON output and auto-fix triple mismatches.
    Returns (ok, cleaned_obj, errors). cleaned_obj includes an 'autofix_notes' list when fixes applied.
    """
    errors: List[str] = []
    cleaned: dict = {}
    autofix_notes: List[str] = []

    # Basic input checks
    if not isinstance(tset, frozenset):
        errors.append(f"tset must be a frozenset, got {type(tset)}")
    if not tset.issubset({"dx", "rx", "px"}):
        errors.append(f"tset contains invalid types: {tset}")
    if not allowed_labels:
        errors.append("allowed_labels is empty for this type-set")


    # Required keys in compact schema
    required = [
        "predicted_relationship",
        "KG_triple",
        "confidence",
    ]
    missing = [k for k in required if k not in obj]
    if missing:
        errors.append(f"Missing keys: {missing}")

    # Extract raw values
    rel = str(obj.get("predicted_relationship", "") or "")
    confidence_raw = obj.get("confidence", 0)
    triple = obj.get("KG_triple", None)

    # Basic validations (label + orientation)
    if rel not in (allowed_labels or []):
        errors.append(f"predicted_relationship '{rel}' not in AllowedLabels {allowed_labels}")


    # Handle KG_triple presence/shape
    head_id = pred_label = tail_id = None
    triple_ok = True
    if not (isinstance(triple, list) and len(triple) == 3):
        errors.append("KG_triple must be a 3-element list")
        triple_ok = False
    else:
        # coerce triple elements to strings for robust comparison/fixes
        try:
            head_id = str(triple[0])
            pred_label = str(triple[1])
            tail_id = str(triple[2])
        except Exception:
            errors.append("KG_triple elements must be convertible to strings")
            triple_ok = False

    # If triple exists, try to autofix small issues
    if triple_ok:
        # 1) fix mismatched predicate label in triple
        if pred_label != rel:
            autofix_notes.append(f"Replaced triple label '{pred_label}' -> '{rel}'")
            pred_label = rel  # overwrite

        triple = [head_id, rel, tail_id]

    # Normalize and validate confidence
    try:
        confidence = int(round(float(confidence_raw)))
    except Exception:
        confidence = 0
        errors.append("confidence is not numeric; set to 0")
    confidence = max(0, min(100, confidence))

    # Build cleaned output
    cleaned.update({
        "predicted_relationship": rel,
        "KG_triple": triple,
        "confidence": confidence,
    })
    if autofix_notes:
        cleaned["autofix_notes"] = autofix_notes

    ok = len(errors) == 0
    return ok, cleaned, errors



def build_edge_record(
    cleaned: dict,
    prompt: str,
    pair_info: dict,   # expects: code1_id, code2_id, type1, type2
    stats: dict,
    *,
    model_id: str,
    config_sig: str,
) -> dict:
    """
    Store a single edge record using the ordered input pair (code1, code2) + the LLM decision.
    No pair_key; no surface pair_type. We keep an 'unordered_type' for grouping and 'orientation' for direction.
    """
    # From validator (already consistent)
    head, _label_from_triple, tail = cleaned["KG_triple"]
    canonical_rel = cleaned["predicted_relationship"]

    # Unordered type-set & pretty string for grouping
    unordered_type = "↔".join(sorted([pair_info["type1"], pair_info["type2"]]))  # e.g., "dx↔px"

    return {
        # Inputs (always the order you prompted)
        "code1_id": pair_info["code1_id"],
        "code2_id": pair_info["code2_id"],
        "type1": pair_info["type1"],
        "type2": pair_info["type2"],
        "unordered_type": unordered_type,       # e.g., "dx↔px"
        "predicted_relationship": canonical_rel,  # canonical (unordered) label chosen by LLM
        "confidence": cleaned["confidence"],
        "head": head,
        "tail": tail,
        "stats_json": json.dumps(stats),
        "prompt_id": prompt_hash(prompt),
        "config_sig": config_sig,
        "model_id": model_id,
    }

# -----------------------
# User-supplied helper functions (exactly as you provided)
# -----------------------

def first_json_block(s: str) -> str:
    """
    Extract the first top-level JSON object substring from a text blob.
    Raises json.JSONDecodeError if none is found.
    """
    depth = 0
    start = -1
    for i, ch in enumerate(s):
        if ch == '{':
            if depth == 0:
                start = i
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0 and start != -1:
                return s[start:i+1]
    raise json.JSONDecodeError("No JSON object found", s, 0)




def flush_edges(df: pd.DataFrame, path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    ext = os.path.splitext(path)[1].lower()
    if ext in {".parquet", ".pq"}:
        df.to_parquet(path, index=False)
    else:
        df2 = df.copy()
        if "reasoning_emb" in df2.columns:
            df2["reasoning_emb"] = df2["reasoning_emb"].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)
        df2.to_csv(path, index=False)

def write_error_jsonl(path: str, obj: dict):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


In [ ]:
#!/usr/bin/env python3
"""
stitch_batch_outputs.py

Looks for:
  saved_files/mimic3/KG_openai/batch/gpt5_mini/B1/...,
  saved_files/mimic3/KG_openai/batch/gpt5_mini/B2/..., ..., B6/...

Each batch folder is expected to contain:
  - input_part_*.jsonl            (original inputs — ignored by stitcher)
  - input_part_*_map.csv          (mapping custom_id -> metadata)  <-- used
  - output_batch_*.jsonl          (batch output from OpenAI)     <-- stitched
"""

import os
import json
import hashlib
from pathlib import Path
from typing import Dict, Any, List
from tqdm import tqdm
import pandas as pd

# -----------------------
# Config - edit if needed
# -----------------------
OUT_DIR = Path("saved_files/mimic4/KG_openai/batch/gpt5_mini")
EDGE_PATH = OUT_DIR / "stitched_edges.csv"   # can switch to .parquet if desired
ERRORS_PATH = OUT_DIR / "stitch_errors.jsonl"
BATCH_SUBDIR_PREFIX = "B"            # B1..B6
MAP_GLOB = "input_part_*_map.csv"    # mapping CSV(s)
OUTPUT_GLOB = "output_batch_*.jsonl" # preferred batch output name pattern
FLUSH_BATCH = 5000


def extract_inner_json_from_response(resp: dict) -> dict:
    """
    Given the response wrapper (resp), return the parsed JSON object contained
    in the assistant output's text field.
    """
    if not isinstance(resp, dict):
        raise ValueError("response must be a dict")
    # Look for output -> content -> text
    for out_item in resp.get("output", []):
        if not isinstance(out_item, dict):
            continue
        for piece in out_item.get("content", []):
            if not isinstance(piece, dict):
                continue
            if piece.get("type") in ("output_text", "text") and "text" in piece:
                return json.loads(piece["text"])
    # Fallback to top-level output_text
    if "output_text" in resp:
        return json.loads(resp["output_text"])
    # Last resort: extract first JSON block from stringified resp
    s = json.dumps(resp, ensure_ascii=False)
    jb = first_json_block(s)
    return json.loads(jb)


def parse_one_response_body(body: Any) -> dict:
    if body is None:
        raise ValueError("empty_body")
    if isinstance(body, dict) and body.get("output"):
        out0 = body["output"][0]
        if isinstance(out0, dict) and out0.get("content"):
            for piece in out0["content"]:
                if not isinstance(piece, dict):
                    continue
                if piece.get("type") in ("output_text", "text") and "text" in piece:
                    return json.loads(piece["text"])
    if isinstance(body, dict) and "output_text" in body:
        return json.loads(body["output_text"])
    if isinstance(body, dict) and "predicted_relationship" in body:
        return body
    s = json.dumps(body, ensure_ascii=False)
    jb = first_json_block(s)
    return json.loads(jb)

def load_edges(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        return pd.DataFrame()
    ext = os.path.splitext(path)[1].lower()
    if ext in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

# -----------------------
# Stitching logic
# -----------------------
def stitch_and_save():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Discover B* folders
    batch_dirs: List[Path] = sorted([p for p in OUT_DIR.iterdir() if p.is_dir() and p.name.startswith(BATCH_SUBDIR_PREFIX)])
    if not batch_dirs:
        raise RuntimeError(f"No batch directories (B*) found under {OUT_DIR}")

    # Build meta map by reading all input_part_*_map.csv files from each B*
    meta: Dict[str, dict] = {}
    for bd in batch_dirs:
        maps = sorted(bd.glob(MAP_GLOB))
        for mp in maps:
            dfm = pd.read_csv(mp, dtype=str, keep_default_na=False)
            if "custom_id" not in dfm.columns:
                raise RuntimeError(f"Map file {mp} missing 'custom_id' column")
            for r in dfm.itertuples(index=False):
                meta[r.custom_id] = r._asdict()

    if not meta:
        print("Warning: no metadata mappings found across batch folders — stitcher will likely fail to map records.")

    # Find output jsonl files: prefer output_batch_*.jsonl; fallback to any .jsonl excluding input_part_
    all_output_paths: List[Path] = []
    for bd in batch_dirs:
        outs = sorted(bd.glob(OUTPUT_GLOB))
        if not outs:
            fallback = [p for p in bd.glob("*.jsonl") if not p.name.startswith("input_part_")]
            outs = sorted(fallback)
        all_output_paths.extend(outs)
    if not all_output_paths:
        raise RuntimeError(f"No output jsonl files found under batch folders in {OUT_DIR}")

    # Load existing edges to dedupe
    edges_df = load_edges(str(EDGE_PATH))
    seen_prompt_ids = set(edges_df["prompt_id"]) if (not edges_df.empty and "prompt_id" in edges_df.columns) else set()

    edge_buffer: List[Dict[str, Any]] = []
    n_errors = 0
    total_processed = 0
    total_added = 0

    for path in all_output_paths:
        print(f"Processing outputs in: {path}")
        with path.open("r", encoding="utf-8") as f:
            for line in tqdm(f, desc=f"Reading {path.name}"):
                total_processed += 1
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                except Exception as e:
                    write_error_jsonl(str(ERRORS_PATH), {"error": "json_decode", "line_preview": line[:300], "exc": str(e), "source_file": str(path)})
                    n_errors += 1
                    continue

                cid = rec.get("custom_id")
                if cid is None:
                    write_error_jsonl(str(ERRORS_PATH), {"error": "missing_custom_id", "record_preview": str(rec)[:300], "source_file": str(path)})
                    n_errors += 1
                    continue

                m = meta.get(cid)
                if not m:
                    write_error_jsonl(str(ERRORS_PATH), {"custom_id": cid, "error": "missing_metadata_map", "source_file": str(path)})
                    n_errors += 1
                    continue

                prompt_id = m.get("prompt_id")
                if prompt_id and prompt_id in seen_prompt_ids:
                    # already stitched
                    continue

                if rec.get("error"):
                    write_error_jsonl(str(ERRORS_PATH), {"custom_id": cid, "error": rec.get("error"), "source_file": str(path)})
                    n_errors += 1
                    continue

                # Extract inner JSON produced by the model (the dict validator expects)
                resp_wrapper = rec.get("response", {})   # the full response object from the API
                try:
                    parsed = extract_inner_json_from_response(resp_wrapper.get("body"))
                except Exception as e:
                    write_error_jsonl(str(ERRORS_PATH), {"custom_id": cid, "error": f"extract_inner_json_failed:{e}", "resp_preview": str(resp_wrapper)[:800], "source_file": str(path)})
                    n_errors += 1
                    continue

                # Recreate the types/labels context to validate
                tset = frozenset({m["type1"], m["type2"]})
                allowed_labels = list(CANONICAL_LABELS_WITH_DESC[tset].keys())
                allowed_labels.append('tr eats')

                ok, cleaned, errs = validate_llm_output(
                    parsed,
                    tset=tset,
                    code1_id=m["code1_id"],
                    code2_id=m["code2_id"],
                    allowed_labels=allowed_labels,
                )

                if not ok:
                    write_error_jsonl(str(ERRORS_PATH), {"custom_id": cid, "error": f"validation:{errs}", "raw_parsed": parsed, "source_file": str(path)})
                    n_errors += 1
                    continue

                try:
                    stats = json.loads(m.get("stats_json")) if m.get("stats_json") else {}
                except Exception:
                    stats = {}
                pair_info = {
                    "code1_id": m.get("code1_id", ""),
                    "type1": m.get("type1", ""),
                    "code2_id": m.get("code2_id", ""),
                    "type2": m.get("type2", ""),
                }

                edge_rec = build_edge_record(
                    cleaned,
                    prompt="",
                    pair_info=pair_info,
                    stats=stats,
                    model_id=m.get("model_id", ""),
                    config_sig=m.get("config_sig", ""),
                )
                edge_rec["prompt_id"] = m.get("prompt_id", "")
                edge_buffer.append(edge_rec)
                seen_prompt_ids.add(m.get("prompt_id", ""))
                total_added += 1

                if len(edge_buffer) >= FLUSH_BATCH:
                    dfb = pd.DataFrame(edge_buffer)
                    if edges_df is None or edges_df.empty:
                        edges_df = dfb
                    else:
                        edges_df = pd.concat([edges_df, dfb], ignore_index=True)
                    flush_edges(edges_df, str(EDGE_PATH))
                    edge_buffer.clear()

    # final flush
    if edge_buffer:
        dfb = pd.DataFrame(edge_buffer)
        if edges_df is None or edges_df.empty:
            edges_df = dfb
        else:
            edges_df = pd.concat([edges_df, dfb], ignore_index=True)
        flush_edges(edges_df, str(EDGE_PATH))
        edge_buffer.clear()

    print("=== Stitching complete ===")
    print(f"Total processed lines: {total_processed}")
    print(f"Total edges added: {total_added}")
    print(f"Total errors logged: {n_errors}")
    print(f"Stitched edges saved to: {EDGE_PATH}")
    print(f"Errors logged to: {ERRORS_PATH}")

if __name__ == "__main__":
    stitch_and_save()
